# Synthetic question sets: build one, then decide how far to trust it

Most teams bring documents with no labeled questions, so there is nothing to score a retriever
against. This notebook builds a question set from the documents themselves, records each
question's gold evidence as a character span, and then tests the set against human labels: on
ContractNLI, do generated labels rank RI #1's retrieval arms in the same order as the real ones?

By the end, you can:

- Explain quote-first generation, and why the gold evidence is a span of the document rather
  than a chunk.
- Read the filter funnel: what each filter removes and what it costs in calls.
- Explain how in-domain null questions are written and checked.
- Read measured difficulty (where BM25 and dense retrieval rank the gold passage) in place of a
  model's difficulty label.
- Say what a generated set supports. On ContractNLI it ranked retrieval arms close to the human
  ranking with plain wording (Kendall's tau 0.86 on nDCG@10, the normalized discounted
  cumulative gain at rank 10) and less well with paraphrased wording (0.57). Its absolute scores ran about 0.14 high, and it was never tested on answer
  models.

With `RUN_EXPENSIVE = False` the notebook reads a reference run built on Vector's on-prem models
and makes no model call. With `RUN_EXPENSIVE = True` it repeats the build and the comparison on
your models over 8 documents. `../../regulatory_policy_qa/04_bring_your_own_corpus.ipynb` runs the
same build on a folder of your own documents.

ContractNLI (Koreeda and Manning, 2021) is licensed CC BY 4.0. The outputs below quote its
documents. The generated questions and answers are model output and are not part of ContractNLI.

## 1. Choose your models and the run mode

A build uses three models. The generator writes statements, questions, and null questions. The
checker answers each question from its gold quote alone and looks for a second passage that
answers it. The judge compares the checker's answer with the generator's reference answer.

The checker has to be a different model from the generator, and a different family is better. A
checker from the generator's own family tends to share its blind spots and its phrasing, so the round trip agrees more often than the answers warrant. The default therefore picks the checker by family:
`gpt-6-luna` when the generator is a Google model, and `gemini-3.5-flash-lite` for any other paid generator. An on-prem generator keeps a free checker: `Qwen3_8-27B` for `Qwen3_6-35B-A3B`, and `Qwen3_6-35B-A3B` otherwise. The generator is your generation model (the cheap tier) and the judge is your judging
model, the shared judge `gemini-3.8-flash` for every paid family. With the default `google` family
that gives a Gemini generator, an OpenAI checker, and a Gemini judge. Set `CHECKER` to any model name to override the default. Naming the judging model makes the checker and the judge the same model, and for a Google generator the funnel then records a same-family check.

The reference run used `Qwen3_8-27B` to generate, `Qwen3_6-35B-A3B` to check, and
`Qwen3_8-Flash-Next` to judge, all with thinking off. They are all Qwen models, so the reference
is a same-family check too.

In [1]:
import sys
import textwrap
from collections import Counter
from pathlib import Path


sys.path.insert(0, str(Path.cwd()))

import sqsets
from aieng.advanced_rag import spend
from aieng.advanced_rag.corpora.chunking import chunk_document, relevant_chunk_ids
from aieng.advanced_rag.evaluation.synthetic.replay import Roles
from aieng.advanced_rag.models import choose_models


# Off reads the reference run. On builds a set on your models and scores the arms.
RUN_EXPENSIVE = False
# Documents your live run covers. The reference covers 28.
LIVE_DOCUMENTS = sqsets.LIVE_DOCUMENTS
# None picks a checker from another family than the generator. Name a model to override.
CHECKER = None

MODELS = choose_models(show=True)
ROLES = Roles.from_models(MODELS, checker=CHECKER)
print(f"your build: generator {ROLES.generator}, checker {ROLES.checker}, judge {ROLES.judge}")
if ROLES.same_family():
    print("  the checker is in the generator's family; the funnel records it as self-checked")

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget unknown (the proxy did not answer)
your build: generator gemini-3.5-flash-lite, checker gpt-6-luna, judge gemini-3.8-flash


## 2. Prior art and why this design

Several libraries generate evaluation questions from documents. The design here takes a part
from most of them and differs in where it records the gold evidence.

| Method | How it generates | Gold evidence | Null questions | Taken here |
| --- | --- | --- | --- | --- |
| RAGAS test-set generator | Builds a knowledge graph of extracted headlines, keyphrases, and entities, then runs single-hop and multi-hop synthesizers with personas | Context text | None | Personas, as user categories; varied query style and length |
| DeepEval `Synthesizer` | Groups similar chunks into a context, scores the context with a critic model, and evolves questions (reasoning, comparative, hypothetical, and others) | Context text | None | Nothing directly, because its default embedder is a paid embedding API, which this bootcamp does not call |
| LlamaIndex `generate_question_context_pairs` | Writes a fixed number of questions per chunk | The chunk's node id | None | The output shape (`queries`, `corpus`, `relevant_docs`), through the Benchmarking Information Retrieval (BEIR) export |
| ARES | Writes queries per passage and keeps one only if a retriever returns its passage first, then fits judges and uses prediction-powered inference with about 150 human labels | The passage | None | Checking a small sample by hand. The retriever filter is not used |
| DataMorgana | Combines weighted user and question categories from a configuration into prompts, then filters (a hosted service) | Document ids | None | The configuration: users, question kinds, phrasings, and weights |
| BEIR | A benchmark format: `corpus.jsonl`, `queries.jsonl`, and a tab-separated file of query relevance judgments (qrels) | Corpus ids | Queries with no qrels row | The export format |

Three choices are custom, each for a stated reason:

- The gold is a character span located in the document, so one set scores RI #1's 600- and
  1,200-character chunkers alike. A chunk id, as in LlamaIndex, belongs to one chunker.
- No retriever decides whether a question is kept. ARES's rank-1 filter keeps the questions one
  retriever already finds, which tilts any comparison toward that retriever. Retrieval ranks are
  recorded as difficulty instead.
- The set includes in-domain nulls, which RI #1's abstention scoring needs and a team corpus does
  not have.

Every call goes through the proxy's response cache, so a re-run costs nothing, and embeddings are
local `bge-small-en-v1.5`.

## 3. Load the reference run

The reference build covers the first 28 ContractNLI test documents. Three are held out for null
questions, and the set targets the other 25. The cell labels what it read, and with a paid model
chosen above it prints what rebuilding on your models would cost.

In [2]:
reference_result, reference_set, reference = sqsets.load_validation()
summary = reference_set.summary()
plain = reference_set.variant("plain")
print()
print(
    f"{summary['questions']} questions over {summary['documents']} documents: {summary['answerable']} answerable, {summary['null']} null"
)
print("by kind:   " + ", ".join(f"{k} {v}" for k, v in summary["by_kind"].items()))
print("by wording: " + ", ".join(f"{k} {v}" for k, v in summary["by_variant"].items()))
print(f"plain set used for scoring: {len(plain.answerable)} answerable, {len(plain.nulls)} null")

validation reference artifact, built with Qwen3_8-27B on Vector's cluster at $0
           1,007 calls, 582,215 prompt + 48,109 completion tokens

204 questions over 25 documents: 176 answerable, 28 null
by kind:   held_out_null 14, near_miss_null 14, single_hop 176
by wording: paraphrase 65, plain 139
plain set used for scoring: 111 answerable, 28 null


## 4. Quote first: the gold evidence is a span

Each question starts from a statement. For each sampled passage, the generator copies out up to
two statements verbatim. Code then finds each quote in the document,
tolerating whitespace and punctuation differences, and records its character offsets. A quote it
cannot find is dropped at no cost. Only then does the generator write a question that the
statement answers, for a sampled user (in-house counsel or a business manager), a question kind,
and a phrasing (a full question or search keywords).

Because the offsets are fixed before the question exists, the gold does not depend on how the
question is worded, and it maps onto any chunking. This is the statement-first design of Know
Your RAG (de Lima et al., 2025), with the statement's position recorded.

In [3]:
docs, _ = sqsets.contract_nli(sqsets.REFERENCE_DOCUMENTS)
texts = {d.doc_id: d.text for d in docs}
examples = [q for q in plain.answerable if q.provenance.get("phrasing") == "natural"][:2]
examples += [q for q in plain.answerable if q.provenance.get("phrasing") == "keywords"][:1]
for q in examples:
    gold = q.evidence[0]
    assert texts[gold.doc_id][gold.start : gold.end] == gold.quote
    print(
        f"{q.query_id}  ({q.provenance['user_category']}, {q.provenance['question_category']}, {q.provenance['phrasing']})"
    )
    print(f"  question   {q.question}")
    print(f"  reference  {q.reference}")
    print(f"  gold       document {gold.doc_id}, characters {gold.start} to {gold.end}")
    print(textwrap.indent(textwrap.fill(gold.quote, 96), "             "))
    print()

question = examples[0]
for width, overlap in ((1200, 200), (600, 100)):
    chunks = [
        c
        for d in docs
        if d.doc_id == question.evidence[0].doc_id
        for c in chunk_document(d.doc_id, d.text, chunk_chars=width, overlap_chars=overlap)
    ]
    gold_chunks = sorted(relevant_chunk_ids(chunks, question.answer_spans()))
    print(f"gold chunks for {question.query_id} at {width:>5,} characters: {gold_chunks}")

sq-982c98e88d99  (business_manager, condition_or_exception, natural)
  question   How long does this agreement last, and is there a way it can be extended?
  reference  It lasts for two years, with a possible one-year extension if the parties are still discussing the Transaction at the end of the second year, or the term of the contract, whichever is greater.
  gold       document 1, characters 11040 to 11280
             This Agreement shall remain in effect for a two-year term (subject to a one year extension if
             the parties are still discussing and considering the Transaction at the end of the second year)
             or the term of the contract, whichever is greater.

sq-6e8a8b32f629  (in_house_counsel, factoid, natural)
  question   What is the duration of the confidentiality obligation for information disclosed during the term?
  reference  indefinitely
  gold       document 1, characters 11315 to 11447
             the parties' duty to hold in confidence Confidentia

The same span maps to one or two chunks depending on the chunker, through the harness's own
overlap rule (`relevant_chunk_ids`). Scoring a new chunker needs no new labels.

## 5. The filter funnel

Every candidate ends in exactly one outcome: accepted, rejected by the generator, or filtered,
with the stage and the reason recorded. The filters run in order of cost:

1. `quote_located`: the quote is found in the document (code, free).
2. `round_trip`: the checker answers from the gold quote alone, and the judge compares that
   answer with the reference. A mismatch or a partial answer drops the question (Alberti et al.,
   2019).
3. `uniqueness`: the checker reads the five nearest non-gold passages. A passage that gives the
   same answer is added to the gold as equivalent, and one that gives a different answer marks
   the question ambiguous and drops it.
4. `deduplicated`: two questions above a `bge-small-en-v1.5` cosine of 0.92 keep one.

The paraphrase variant has its own funnel: a question needs terms it shares with its evidence
(`has_leak`), the rewrite has to reduce them (`leak_reduced`), and it has to pass the round trip
again.

In [4]:
funnel = reference_set.meta["funnel"]
print(f"{'kind':<16}{'stage':<18}{'entered':>8}{'passed':>8}{'rate':>7}")
for row in funnel["stages"]:
    print(f"{row['kind']:<16}{row['stage']:<18}{row['entered']:>8}{row['passed']:>8}{row['acceptance_rate']:>7.0%}")
reasons = Counter()
for reason, count in funnel["drop_reasons"].items():
    stage, _, detail = reason.partition(": ")
    reasons[f"{stage}: {detail.split(':')[0]}"[:60]] += count
print()
print("why candidates were dropped:")
for reason, count in reasons.most_common(8):
    print(f"  {count:>3}  {reason}")
print()
for row in reference_set.meta["usage"]:
    print(
        f"{row['model']:<20}{row['cold_requests']:>5} calls  {row['cold_prompt_tokens']:>8,} prompt + {row['cold_completion_tokens']:>6,} completion tokens"
    )

kind            stage              entered  passed   rate
single_hop      quote_located          138     137    99%
single_hop      question_written       137     137   100%
single_hop      round_trip             137     122    89%
single_hop      uniqueness             122     115    94%
single_hop      deduplicated           115     111    97%
near_miss_null  null_written            54      54   100%
near_miss_null  null_check              54      14    26%
held_out_null   quote_located           32      32   100%
held_out_null   null_written            28      28   100%
held_out_null   null_check              28      15    54%
near_miss_null  deduplicated            14      14   100%
held_out_null   deduplicated            14      14   100%
paraphrase      has_leak               111      82    74%
paraphrase      leak_reduced            82      74    90%
paraphrase      round_trip              74      65    88%

why candidates were dropped:
   50  null_check: corpus answers it (yes)

Of 138 quoted statements, 111 became accepted single-hop questions. One quote was not found in its document. The round trip removed the most (15), mostly with answers the judge called partial: the checker's answer left out a condition the
reference included. Uniqueness removed 7 questions that another passage answered differently, and
deduplication 4. The checker made more calls than the generator (415 against 391) and read the
most tokens, because the round trip, uniqueness, and the null checks all send it passages. That makes the checker the main cost driver, so a cheaper checker saves the most.

## 6. Null questions

RI #1 scores abstention on questions the documents do not answer, and a team corpus has none.
The build writes two in-domain kinds, both in the corpus's own vocabulary:

- Near-miss nulls change the one detail of an accepted question that selects its evidence, such
  as a party, a period, or a condition.
- Held-out nulls are written from documents kept out of the index, so their answer is absent by
  construction.

Each one is kept only if the checker, shown the top 10 passages from BM25 and from dense
retrieval over the indexed documents, finds no answer (the corpus-level check of CRUMQs, Liu et
al., 2026). The default share is 20% of the set.

In [5]:
by_id = {q.query_id: q for q in reference_set}
near = [q for q in reference_set.nulls if q.kind == "near_miss_null"]
held = [q for q in reference_set.nulls if q.kind == "held_out_null"]
for q in near[:2]:
    source = by_id.get(q.provenance.get("source_question"))
    print(f"source     {source.question if source else '(source not kept)'}")
    print(f"near miss  {q.question}")
    print(f"changed    {q.provenance.get('changed')}")
    print()
print(f"held out   {held[0].question}  (written from held-out document {held[0].provenance['held_out_document']})")
print()
for row in funnel["stages"]:
    if row["stage"] == "null_check":
        print(f"{row['kind']:<16} null check kept {row['passed']} of {row['entered']} ({row['acceptance_rate']:.0%})")

source     What are the permissible methods for delivering notices under this agreement?
near miss  What are the permissible methods for delivering notices under this agreement if the recipient's electronic mail address is invalid?
changed    added a condition regarding invalid email

source     obligations expire earlier of
near miss  obligations expire later of
changed    changed 'earlier' to 'later'

held out   arbitration tribunal composition exception  (written from held-out document 23)

near_miss_null   null check kept 14 of 54 (26%)
held_out_null    null check kept 15 of 28 (54%)


The null check is the strictest filter in the build. It kept 14 of 54 near-miss rewrites: in
most of the rest the changed detail did not change the answer, or another clause still answered
it. Held-out documents passed more often (15 of 28, 14 after deduplication) because their text is absent from the index,
but non-disclosure agreements repeat each other's clauses, so about half were still answered by another contract's
wording. That repetition is the reason the check runs against the whole indexed corpus rather
than against the source document alone.

## 7. Measured difficulty

Asking the generator to label each question easy, medium, or hard gave "medium" for 49 of 50. Each question here carries numbers instead, measured with the
generator's own probe index (1,200-character chunks, scoped to the question's document): the
rank at which BM25 and dense retrieval place the gold chunk, and the terms the question shares
with its evidence that are rare in the corpus (`leaked_terms`). These are recorded and never used
to filter.

In [6]:
def bucket(rank):
    """Return the rank band a gold chunk falls in."""
    if rank is None:
        return "11+"
    return "1" if rank == 1 else "2-3" if rank <= 3 else "4-10"


order = ["1", "2-3", "4-10", "11+"]


def cells(counts):
    """Return one right-aligned count per rank band."""
    return "".join(f"{counts.get(k, 0):>6}" for k in order)


print("questions per rank of the gold chunk, and leaked terms per question")
print(f"{'':<12}{'':>5}  {'BM25 rank':^24}  {'dense rank':^24}")
print(
    f"{'variant':<12}{'n':>5}  "
    + "".join(f"{k:>6}" for k in order)
    + "  "
    + "".join(f"{k:>6}" for k in order)
    + f"{'leaked':>9}"
)
for name in ("plain", "paraphrase"):
    group = [q for q in reference_set.answerable if q.variant == name]
    bm25 = Counter(bucket(q.labels.get("bm25_rank")) for q in group)
    dense = Counter(bucket(q.labels.get("dense_rank")) for q in group)
    leak = sum(q.labels.get("leak_count", 0) for q in group) / len(group)
    print(f"{name:<12}{len(group):>5}  {cells(bm25)}  {cells(dense)}{leak:>9.2f}")

questions per rank of the gold chunk, and leaked terms per question
                          BM25 rank                 dense rank       
variant         n       1   2-3  4-10   11+       1   2-3  4-10   11+   leaked
plain         111      87    20     4     0      91    15     5     0     1.64
paraphrase     65      32    20    13     0      52     9     4     0     0.14


Most plain questions are easy for BM25 within their own contract: it puts the gold chunk first for 87 of 111. A contract has a median of 9 chunks of this size (3 to 24), so a
scoped search saturates quickly. The arms in the next section separate mainly on nDCG@10, which rewards
the exact rank. The paraphrase variant cuts the leaked terms from 1.64 per question to 0.14, and
the gold falls in BM25's ranking as a result: first for 32 of 65 paraphrased questions.

## 8. Do generated labels rank the arms like human labels?

The test scores RI #1's retrieval arms twice over the same 25 documents: once against
ContractNLI's 219 human-labeled answerable queries, and once against the generated questions.
The eight arms are BM25, dense `bge-small-en-v1.5`, hybrid reciprocal rank fusion at dense
weights 0.5 and 0.2, dense with a MiniLM cross-encoder reranker, and a shuffled control, all at
1,200-character chunks, plus BM25 and dense at 600-character chunks. Retrieval is scoped to each
question's document, as in RI #1.

Two statistics compare the rankings (van Elburg et al., 2025, use the first):

- Kendall's tau between the arm order under human labels and under generated labels. One means the same order, and zero means no relation.
- Sign agreement: for each pair of arms whose paired bootstrap interval excludes zero under human
  labels, whether generated labels give the difference the same sign. A significant reversal is
  a pair both label sets separate, in opposite directions.

In [7]:
rows = sqsets.arm_table(reference)
for variant in ("plain", "paraphrase"):
    print(f"{variant} wording, nDCG@10")
    print(f"  {'arm':<24}{'human':>8}{'rank':>6}{'generated':>11}{'rank':>6}")
    for row in rows:
        if row["variant"] == variant:
            print(
                f"  {row['arm']:<24}{row['real']:>8.3f}{row['real_rank']:>6}{row['generated']:>11.3f}{row['generated_rank']:>6}"
            )
    print()

print(f"{'variant':<12}{'metric':<10}{'tau':>6}{'signs agree':>14}{'reversals':>11}{'mean gap':>10}")
for variant in ("plain", "paraphrase"):
    for name in ("ndcg@10", "recall@3", "mrr"):
        r = reference["agreement"][variant]["all_arms"][name]
        signs = (
            f"{round(r['sign_agreement_on_significant'] * r['real_significant_pairs'])}/{r['real_significant_pairs']}"
        )
        print(
            f"{variant:<12}{name:<10}{r['kendall_tau']:>6.2f}{signs:>14}{len(r['significant_reversals']):>11}{r['mean_gap']:>+10.3f}"
        )
for pair in reference["agreement"]["paraphrase"]["all_arms"]["ndcg@10"]["significant_reversals"]:
    print(f"paraphrase reversal on nDCG@10: {pair}")

plain wording, nDCG@10
  arm                        human  rank  generated  rank
  dense+rerank@1200          0.824     1      0.941     1
  hybrid-rrf-0.5@1200        0.789     2      0.918     2
  hybrid-rrf-0.2@1200        0.778     3      0.903     3
  bm25@1200                  0.771     4      0.890     6
  dense@1200                 0.755     5      0.895     4
  dense@600                  0.653     6      0.892     5
  bm25@600                   0.640     7      0.832     7
  shuffled-control@1200      0.538     8      0.568     8

paraphrase wording, nDCG@10
  arm                        human  rank  generated  rank
  dense+rerank@1200          0.824     1      0.900     1
  hybrid-rrf-0.5@1200        0.789     2      0.849     4
  hybrid-rrf-0.2@1200        0.778     3      0.787     5
  bm25@1200                  0.771     4      0.764     6
  dense@1200                 0.755     5      0.876     2
  dense@600                  0.653     6      0.855     3
  bm25@600          

With plain wording, generated labels put the arms in almost the human order: tau is 0.86 on
nDCG@10, and of the 22 arm pairs the human labels separate, generated labels give 21 the same
sign and reverse none. The reranker comes first and the shuffled control last under both. Tau is
lower on recall@3 (0.64) and on mean reciprocal rank (MRR, 0.50), where several arms sit within a few hundredths of each
other and small shifts reorder them.

Paraphrased wording agrees less: tau falls to 0.57 and two pairs reverse with both intervals
excluding zero. The rewrite removes the terms a question shares with its evidence. That lifts both
dense arms above both hybrids, and 600-character dense from sixth to third. ContractNLI's human
queries share vocabulary with their evidence, so the paraphrase adds a lexical penalty the real
queries do not have. Plain wording is the default for that reason. A corpus whose real users do
not echo the document's wording could reverse this choice, and ContractNLI has no such queries to
test it.

Two biases hold under both wordings:

- Generated labels score every arm higher, by 0.14 nDCG@10 on average with plain wording. A
  generated question was written from one short span and checked to be answerable from it, which
  makes it easier than a human query. Compare arms by rank or by paired difference on the same
  set, never an absolute score against a number from another set.
- Generated labels overrate small chunks. Under human labels 600-character dense trails
  1,200-character dense by 0.10 nDCG@10, and under plain generated labels the two are nearly tied. A
  generated question has one gold span of a sentence or two, while a human ContractNLI query has
  about two gold spans, so a small chunk loses less.

## 9. Explore: write each question from two statements

The chunk-size bias suggests a fix: give each generated question more gold, closer to a human
query's. This single run (one seed, one configuration, on-prem models at 15 requests per
minute or fewer) tests two ways, both scored on the same arms against the same human labels:

- Pairs: the generator writes one question that needs two statements from the same passage, so
  each question has two gold spans. The round trip answers from both quotes, and a second round
  trip from the first quote alone is recorded and does not filter.
- Widened spans: each plain question's gold span is widened around itself to the mean total gold
  length of the human queries. This costs no model call.

In [8]:
explore = sqsets.load_explore().value
real_gold = explore["real"]["gold"]
print()
print(
    f"human queries: {real_gold['spans_per_query']} spans and {real_gold['gold_chars_per_query']:.0f} gold characters per query"
)
pair_set = explore["pair_set"]
print(
    f"pairs: {pair_set['pairs_tried']} statement pairs tried, {pair_set['summary']['answerable']} accepted; "
    f"statement A alone answered {pair_set['first_statement_alone_answers']:.0%} of the accepted questions"
)
print()
human = explore["real"]["mean"]["ndcg@10"]
human_rank = sqsets.ranks(human)
print(
    f"{'labels':<10}{'spans':>7}{'gold chars':>12}{'tau nDCG@10':>13}{'reversals':>11}{'mean gap':>10}{'dense@600 rank':>16}{'bm25@600 rank':>15}"
)
print(
    f"{'human':<10}{real_gold['spans_per_query']:>7}{real_gold['gold_chars_per_query']:>12.0f}{'':>13}{'':>11}{'':>10}{human_rank['dense@600']:>16}{human_rank['bm25@600']:>15}"
)
for name, row in explore["label_sets"].items():
    r = row["all_arms"]["ndcg@10"]
    rank = sqsets.ranks(r["generated_mean"])
    print(
        f"{name:<10}{row['gold']['spans_per_query']:>7}{row['gold']['gold_chars_per_query']:>12.0f}{r['kendall_tau']:>13.2f}"
        f"{len(r['significant_reversals']):>11}{r['mean_gap']:>+10.3f}{rank['dense@600']:>16}{rank['bm25@600']:>15}"
    )

explore-adjacent-statements reference artifact, built with Qwen3_8-27B on Vector's cluster at $0
                            389 calls, 169,112 prompt + 24,889 completion tokens

human queries: 1.86 spans and 401 gold characters per query
pairs: 67 statement pairs tried, 50 accepted; statement A alone answered 10% of the accepted questions

labels      spans  gold chars  tau nDCG@10  reversals  mean gap  dense@600 rank  bm25@600 rank
human        1.86         401                                                 6              7
plain        1.12         281         0.86          0    +0.136               5              7
pairs        2.08         499         0.79          0    +0.121               5              7
widened      1.12         448         0.86          0    +0.131               6              7


Widening each plain question's gold span to at least the
human queries' mean gold length (401 characters) puts 600-character dense back in sixth place, where
human labels put it, keeps tau at 0.86, and gives all 22 human-separated pairs the human sign on nDCG@10 (on MRR one
pair reverses).

The pairs need both statements: statement A alone answered 10% of the 50 accepted questions, and
their gold averages 2.1 spans, close to the human 1.9. They also restore part of the chunk-size
gap. BM25 at 600 characters trails BM25 at 1,200 by 0.10 nDCG@10 under pair labels, against 0.06
under plain labels and 0.13 under human labels. Their tau is lower, though (0.79), because
1,200-character dense moves above the 0.2-weight hybrid. The pair run cost 389 on-prem calls.

Scoped to what ran (25 ContractNLI documents, one seed, `Qwen3_8-27B` writing from two statements
of one 1,200-character passage), pairs did not raise rank agreement. Two things would have to be
true for them to win: the lower tau comes from the smaller set (50 questions against 111) rather
than from the pairing, and pairs drawn from different passages match the spread of human gold
better. The cheapest test of the first is to score a random 50 of the plain questions and compare
its tau, which needs no model call. Widened spans are the cheaper correction to try first on
another corpus.

## 10. Run it on your models

With `RUN_EXPENSIVE = True`, this cell builds a set over the first `LIVE_DOCUMENTS` ContractNLI
documents with your three models and scores the arms under human and generated labels, as the
reference did on 28. The cell prints the estimate before it runs and stops if it is over 10% of
your key's remaining budget. With the flag off it prints the reference and what your run would
cost.

Eight documents give about 30 accepted questions, so expect a noisier tau than the reference's:
one swapped pair among eight arms moves tau by 0.07.

In [9]:
mine_result, mine_set, mine = sqsets.load_validation(
    models=MODELS, roles=ROLES, documents=LIVE_DOCUMENTS, live=RUN_EXPENSIVE
)
label = "your run" if mine_result.origin != "reference" else "reference run (set RUN_EXPENSIVE = True for yours)"
print()
print(label)
mine_funnel = mine_set.meta["funnel"]
answerable = {row["stage"]: row for row in mine_funnel["stages"] if row["kind"] == "single_hop"}
print(
    f"  {len(mine_set.variant('plain').answerable)} answerable and {len(mine_set.nulls)} null questions over {mine['documents_scored']} scored documents"
)
print("  single-hop acceptance: " + ", ".join(f"{s} {r['acceptance_rate']:.0%}" for s, r in answerable.items()))
for name in ("ndcg@10", "recall@3", "mrr"):
    r = mine["agreement"]["plain"]["all_arms"][name]
    print(
        f"  plain wording, {name:<9} tau {r['kendall_tau']:.2f}, significant reversals {len(r['significant_reversals'])}, mean gap {r['mean_gap']:+.3f}"
    )

validation reference artifact, built with Qwen3_8-27B on Vector's cluster at $0
           1,007 calls, 582,215 prompt + 48,109 completion tokens
           (the reference covers 28 documents; your build covers 8)
           your build: gemini-3.5-flash-lite generates, gpt-6-luna checks, gemini-3.8-flash judges; about 287 calls, est. $0.06 (set RUN_EXPENSIVE = True to build)

reference run (set RUN_EXPENSIVE = True for yours)
  111 answerable and 28 null questions over 25 scored documents
  single-hop acceptance: quote_located 99%, question_written 100%, round_trip 89%, uniqueness 94%, deduplicated 97%
  plain wording, ndcg@10   tau 0.86, significant reversals 0, mean gap +0.136
  plain wording, recall@3  tau 0.64, significant reversals 0, mean gap +0.187
  plain wording, mrr       tau 0.50, significant reversals 0, mean gap +0.145


## 11. When to trust a generated set

Scoped to what ran (ContractNLI, 25 scored documents, one generation seed, one configuration, a
Qwen generator checked within its own family), a generated set supports these uses:

- Ranking retrieval configurations: chunkers, embedders, hybrid weights, rerankers. With plain
  wording the order matched the human order closely (tau 0.86 on nDCG@10) and no significant pair
  reversed.
- Paired differences between two arms on the same set, with a bootstrap interval. A difference
  whose interval excludes zero on generated labels had the human sign in 21 of 22 pairs.

It does not support these:

- Absolute scores. Generated labels ran about 0.14 nDCG@10 high. Report a rank or a paired
  difference, and never compare a score on a generated set with a score on a human set.
- Ranking answer models or prompts. This validation scored retrieval only. Van Elburg et al.
  (2025) found that generated sets rank generator models unreliably, sometimes in reverse, and
  nothing here tests otherwise.
- Small chunks against large ones, without a correction. Generated labels favor small chunks,
  and section 9 measures two corrections.

Before relying on a set for a decision, read 20 to 30 of its questions with someone who knows
the documents, and count how many have a correct reference answer. That count bounds how far the
labels can be trusted, the same logic ARES applies with prediction-powered inference. Keep plain
wording unless your users' questions are known to avoid the documents' vocabulary, and keep the
checker in a different family from the generator, as the default does.

In [10]:
print(spend.report())

this run, estimated   est. $0.00 over 0 models
